# TAREA 6
María Fernanda Gómez Flores  
Carlos Hassan Hernández Rohan  
Portafolios de Inversión

## PUNTO A

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

### 1. MÍNIMA SEMIVARIANZA

In [ ]:
# Descarga de Datos
prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
                     start='2016-01-01', end='2026-10-15')['Close']

prices.isna().sum()

In [ ]:
# Calculo de rendimientos
rets=prices.pct_change().dropna()

In [ ]:
# Comprobar datos
print(prices.index.min(),prices.index.max())
print(prices.isna().sum())

In [ ]:
def opt_min_semivar(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular matriz de correlación
    corr = rets.corr()

    #Calcular downside risk
    below_zero = (rets[rets<0]).fillna(0)
    downside = np.array(below_zero.std())

    #Calcular matriz de semivarianza
    n_assets = len(prices.columns)
    semivar_matrix = (downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*corr

    # Funcion.
    semivar = lambda w: w.T @ semivar_matrix @ w

    #Variables iniciales (pesos)
    w_0 = np.ones(n_assets) / n_assets

    # Restricciones.
    restr = lambda w: sum(w) - 1

    #rangos de las variables de decisión
    rangos = [(0,1)] * n_assets

    # tolerancia
    tol = 1e-12

    #Optimizar
    results = minimize(fun=semivar, x0=w_0, bounds=rangos,
                       constraints={'fun': restr, 'type': 'eq'}, tol=tol)

    if not results.success:
        print(results.message)

    return results.x

In [ ]:
# Verificar resultados de la función
w_min_semivar=opt_min_semivar(prices)
dict(zip(prices.columns,w_min_semivar))

#### Comprobación Montecarlo

In [ ]:
below_zero=(rets[rets<0]).fillna(0)
below_zero

In [ ]:
downside=np.array(below_zero.std())
downside

In [ ]:
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix=(downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*rets.corr()
semivar_matrix

In [ ]:
semivar_port=lambda w: w.T @ semivar_matrix @ w

In [ ]:
# Obtener semivarianza mínima
semivar_port(w_min_semivar)

In [ ]:
# Simular portafolios con montecarlo
np.random.seed(42)
n_sim=10000
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [ ]:
# Obtener la mínima semivarianza
semivars_port=[semivar_port(w_i) for w_i in w_random]
np.min(semivars_port)

In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza
w_semivar=w_random[np.argmin(semivars_port)]
dict(zip(prices.columns, w_semivar))

In [ ]:
# Comparar resultados
comparacion=pd.DataFrame({
    'SciPy':w_min_semivar*100,
    'Montecarlo':w_semivar*100
},index=prices.columns)

comparacion

In [ ]:
# Comparar semivarianza mínima
resultados_semivar=pd.DataFrame({
    'Semivarianza':[semivar_port(w_min_semivar),np.min(semivars_port)]
},index=['SciPy','Montecarlo'])
resultados_semivar

**Conclusión**  
En mínima semivarianza obtuvimos 0.0000354171 con SciPy y 0.0000356383 con Montecarlo, entonces los resultados quedaron bastante cercanos. En SciPy el mayor peso fue KO con 32.80%, seguido de PG con 22.27% y WMT con 19.31%. En Montecarlo también KO tuvo el mayor peso con 34.35%. SciPy encontró una semivarianza un poco menor porque Montecarlo solo compara los portafolios que simulamos.

### 2. MÁXIMO OMEGA

In [ ]:
def opt_max_omega(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular downside y upside risk
    below_zero=rets[rets<0].fillna(0)
    above_zero=rets[rets>0].fillna(0)
    upside_risk=np.array(above_zero.std())
    downside_risk=np.array(below_zero.std())

    # Calcular Omega individual
    omega=upside_risk/downside_risk

    # Funcion
    func_omega=lambda w: -sum(w*omega)

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    n_assets=len(prices.columns)
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-12

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_omega,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    if not results.success:
        print(results.message)

    return results.x

In [ ]:
# Obtener los pesos
w_max_omega=opt_max_omega(prices)
dict(zip(prices.columns,w_max_omega))

#### Comprobación Montecarlo

In [ ]:
below_zero=rets[rets<0].fillna(0)
above_zero=rets[rets>0].fillna(0)

In [ ]:
upside_risk=np.array(above_zero.std())
upside_risk

In [ ]:
downside_risk=np.array(below_zero.std())
downside_risk

In [ ]:
omega=upside_risk/downside_risk
omega

In [ ]:
# Obtener el maximo omega
func_omega=lambda w: sum(w*omega)

In [ ]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [ ]:
# Obtener el maximo omega
omegas_port=[func_omega(w_i) for w_i in w_random]
np.max(omegas_port)

In [ ]:
# Obtener los pesos del portafolio con maximo omega
w_omega_mc=w_random[np.argmax(omegas_port)]
dict(zip(rets.columns,w_omega_mc))

In [ ]:
# Comparar resultados
comparacion_omega=pd.DataFrame({
    'SciPy':w_max_omega*100,
    'Montecarlo':w_omega_mc*100
},index=prices.columns)
comparacion_omega

In [ ]:
# Comparar Omega máximo
resultados_omega=pd.DataFrame({
    'Omega':[func_omega(w_max_omega),np.max(omegas_port)]
},index=['SciPy','Montecarlo'])
resultados_omega

**Conclusión**  
En máximo Omega cambió mucho la distribución, porque con SciPy WMT quedó con prácticamente 100% del portafolio, mientras que en Montecarlo obtuvo 90.36%. El Omega máximo fue 1.070401 con SciPy y 1.057478 con Montecarlo. Esto tiene sentido porque WMT fue el activo con el Omega individual más alto, pero el portafolio quedó muy concentrado en una sola empresa. Aquí estamos usando la fórmula de Omega que vimos en clase.

### 3. SEMIVARIANZA OBJETIVO

In [ ]:
# Descarga de Precios del Benchmark
benchmark = yf.download('KXI', start='2016-01-01', end='2026-10-15')['Close']
benchmark

In [ ]:
# Obtener rendimientos del benchmark
rets_benchmark=benchmark.pct_change().dropna()
rets_benchmark

In [ ]:
# Comprobar fechas en común
fechas=rets.index.intersection(rets_benchmark.index)
print(len(rets),len(rets_benchmark),len(fechas))
print(fechas.min(),fechas.max())
print(rets.loc[fechas].isna().sum())
print(rets_benchmark.loc[fechas].isna().sum())

In [ ]:
def opt_target_semivar(prices: pd.DataFrame, benchmark: pd.Series):
    #Calcular los rendimientos
    rets=prices.pct_change().dropna()
    rets_benchmark=benchmark.pct_change().dropna()

    # Sustraer rendimiento del benchmark de rendimientos individuales
    alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()

    # Separar los rendimientos por debajo de cero
    below_zero_t=(alpha[alpha<0]).fillna(0)

    # Calcular target downside risk
    downside_risk_t=np.array(below_zero_t.std())

    # Calcular la matriz de target semivariance
    corr=alpha.corr()
    n_assets=len(prices.columns)
    semivar_matrix=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*corr

    # Funcion
    func_target=lambda w: w.T @ semivar_matrix @ w

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-12

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_target,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    if not results.success:
        print(results.message)

    return results.x

In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_target_semivar=opt_target_semivar(prices,benchmark)
dict(zip(prices.columns,w_target_semivar))

#### Comprobación Montecarlo

In [ ]:
# Sustraer rendimiento del benchmark de rendimientos individuales
alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()
alpha

In [ ]:
# Separar los rendimientos por debajo de cero
below_zero_t=(alpha[alpha<0]).fillna(0)
below_zero_t

In [ ]:
# Calcular target downside risk
downside_risk_t=np.array(below_zero_t.std())
downside_risk_t

In [ ]:
# Calcular la matriz de target semivariance
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix_t=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*alpha.corr()
semivar_matrix_t

In [ ]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [ ]:
# Obtener la mínima target semivariance
func_target = lambda w: w.T @ semivar_matrix_t @ w
mts=[func_target(w_i)for w_i in w_random]
np.min(mts)

In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_tarjet_semivar = w_random[np.argmin(mts)]
dict(zip(rets.columns, w_tarjet_semivar))

In [ ]:
# Comparar resultados
comparacion_target=pd.DataFrame({
    'SciPy':w_target_semivar*100,
    'Montecarlo':w_tarjet_semivar*100
},index=prices.columns)

comparacion_target

In [ ]:
# Comparar semivarianza objetivo
resultados_target=pd.DataFrame({
    'Semivarianza objetivo':[func_target(w_target_semivar),np.min(mts)]
},index=['SciPy','Montecarlo'])
resultados_target.apply(lambda x: x.map(lambda v: f'{v:.10f}'))

**Conclusión**  
En semivarianza objetivo utilizamos KXI como benchmark. Con SciPy obtuvimos 0.0000078267 y con Montecarlo 0.0000078767, por lo que SciPy encontró una semivarianza objetivo ligeramente menor. En SciPy los activos con mayor peso fueron KO con 33.60% y PG con 27.47%; en Montecarlo fueron KO con 35.20% y PG con 25.39%. Los pesos cambiaron respecto a mínima semivarianza porque ahora tomamos como referencia el rendimiento de KXI.

## PUNTO B

Seleccionamos cinco activos de consumo básico: PG, COST, KO, WMT y CLX. Utilizamos KXI como benchmark porque corresponde al sector de consumo básico. Vamos a aplicar las tres funciones del Punto A.

In [ ]:
# Descarga de Datos
prices_b=yf.download(['PG','COST','KO','WMT','CLX'],
                     start='2016-01-01',end='2026-10-15')['Close']
prices_b.isna().sum()

In [ ]:
# Calculo de rendimientos
rets_b=prices_b.pct_change().dropna()
rets_b

### 1. MÍNIMA SEMIVARIANZA

In [ ]:
# Obtener los pesos
w_min_semivar_b=opt_min_semivar(prices_b)
dict(zip(prices_b.columns,w_min_semivar_b))

In [ ]:
# Calcular matriz de correlación
corr_b=rets_b.corr()
corr_b

In [ ]:
below_zero_b=(rets_b[rets_b<0]).fillna(0)
below_zero_b

In [ ]:
downside_b=np.array(below_zero_b.std())
downside_b

In [ ]:
# Calcular matriz de semivarianza
n_assets_b=len(prices_b.columns)
semivar_matrix_b=(downside_b.reshape(n_assets_b,1) @ downside_b.reshape(1,n_assets_b))*corr_b
semivar_matrix_b

In [ ]:
# Obtener semivarianza mínima
semivar_port_b=lambda w: w.T @ semivar_matrix_b @ w
semivar_port_b(w_min_semivar_b)

### 2. MÁXIMO OMEGA

In [ ]:
# Obtener los pesos
w_max_omega_b=opt_max_omega(prices_b)
dict(zip(prices_b.columns,w_max_omega_b))

In [ ]:
# Calcular upside risk
above_zero_b=rets_b[rets_b>0].fillna(0)
upside_risk_b=np.array(above_zero_b.std())
upside_risk_b

In [ ]:
# Calcular Omega individual
omega_b=upside_risk_b/downside_b
omega_b

In [ ]:
# Obtener máximo Omega
func_omega_b=lambda w: sum(w*omega_b)
func_omega_b(w_max_omega_b)

### 3. SEMIVARIANZA OBJETIVO

In [ ]:
# Usar el mismo benchmark del Punto A
benchmark_b=benchmark

In [ ]:
# Obtener rendimientos del benchmark
rets_benchmark_b=rets_benchmark

In [ ]:
# Verificar fechas en común
fechas_b=rets_b.index.intersection(rets_benchmark_b.index)
print(len(rets_b),len(rets_benchmark_b),len(fechas_b))
print(prices_b.isna().sum())

In [ ]:
# Obtener los pesos
w_target_semivar_b=opt_target_semivar(prices_b,benchmark_b)
dict(zip(prices_b.columns,w_target_semivar_b))

In [ ]:
# Sustraer rendimiento del benchmark
alpha_b=rets_b.sub(rets_benchmark_b.squeeze(),axis=0).dropna()
alpha_b

In [ ]:
# Calcular target downside risk
below_zero_t_b=(alpha_b[alpha_b<0]).fillna(0)
downside_risk_t_b=np.array(below_zero_t_b.std())
downside_risk_t_b

In [ ]:
# Calcular matriz de target semivariance
semivar_matrix_t_b=(downside_risk_t_b.reshape(n_assets_b,1) @ downside_risk_t_b.reshape(1,n_assets_b))*alpha_b.corr()
semivar_matrix_t_b

In [ ]:
# Obtener semivarianza objetivo
func_target_b=lambda w: w.T @ semivar_matrix_t_b @ w
func_target_b(w_target_semivar_b)

### 4. COMPARACIÓN DE RESULTADOS

In [ ]:
# Comparar los pesos de cada método
pesos_b=pd.DataFrame({
    'Mínima Semivarianza':w_min_semivar_b*100,
    'Máximo Omega':w_max_omega_b*100,
    'Semivarianza Objetivo':w_target_semivar_b*100
},index=prices_b.columns)
pesos_b

In [ ]:
# Comparar los valores óptimos
resultados_b=pd.DataFrame({
    'Método':['Mínima Semivarianza','Máximo Omega','Semivarianza Objetivo'],
    'Valor óptimo':[semivar_port_b(w_min_semivar_b),func_omega_b(w_max_omega_b),func_target_b(w_target_semivar_b)]
})
resultados_b['Valor óptimo']=resultados_b['Valor óptimo'].map(lambda x:f'{x:.10f}')
resultados_b

**Conclusión**  
Al optimizar los cinco activos, en mínima semivarianza obtuvimos 0.0000354575. El mayor peso quedó en KO con 33.31%, seguido de PG con 23.48% y WMT con 19.12%. En máximo Omega el valor fue 1.0704014710 y prácticamente todo el peso quedó en WMT con 100%, entonces este método fue el menos diversificado. Para semivarianza objetivo, utilizando KXI como benchmark, obtuvimos 0.0000083998 y los mayores pesos fueron KO con 37.25% y PG con 29.65%. Podemos ver que los métodos dan distribuciones diferentes porque cada uno busca optimizar algo distinto: reducir pérdidas, maximizar Omega o disminuir el riesgo de quedar por debajo del benchmark.